##Introduction to UDF
####Spark UDF
 1. Scalar Python UDF
 2. Pandas Vectorized UDF
 3. UDTF
####Scalar Python UDF
 1. User-defined functions 
 2. Take or return Python objects
 3. Operate one row at a time
 4. Serialized/Deserialized by pickle or Arrow 



####1. How to define and use a Python UDF

1.1 Define a UDF\
 Create a UDF with the following functionality
 * Input -> member_id
 * Output -> Guest/Member

In [0]:
from pyspark.sql.functions import udf
#udf is an anotation which allows us to define a User-Defined Function(UDF) in python
#Arrow and Pickle are serialization library that converts data (objects, DataFrames, etc.) into a format that can be: stored (file, disk) or transferred (network, between processes)

@udf(returnType='string', useArrow=True)
def member_type_udf(id: str):
    return 'Guest' if id==0 else 'Member'

#Spark User-Defined Functions are defined for the current session, and they can be used in the current session only. They cannot be used across different sessions

1.2 Use a UDF in Dataframe Transformations

In [0]:
member_df = spark.table("dev.spark_db.members")

result_df = member_df.withColumn("member_type", member_type_udf("member_id"))

result_df.limit(3).display()

1.3. Register UDF for use in Spark SQL engine

In [0]:
spark.udf.register("member_type_udf", member_type_udf)

1.4 UDF from Spark SQL

Once the udf is register we can use it in spark sql without any "Cannot resolve routine" error

In [0]:
%sql

 select *, member_type_udf(member_id) as member_type
 from dev.spark_db.members
 limit 3

1.5 UDF in Expressions

In [0]:
from pyspark.sql.functions import expr

member_df = spark.table("dev.spark_db.members")

result_df = member_df.withColumn("member_type", expr("member_type_udf(member_id)"))

result_df.limit(3).display()

%md
----------------------------------------------------------------------------------------------------------------------------------------------
###Pandas UDF (Vectorized UDF)
 1. User-defined functions 
 2. Take or return Pandas Series/DataFrame
 3. Operate block by block (Vectorized) - Block by block means you take a block of IDs or block of input means a array of input, process the entire array and then return the result as a array the entire block
 4. Serialized/Deserialized by Arrow

 Note: You must know Pandas to work with the data inside the function

####1. How to define and use a Pandas UDF

1.1 Create a Pandas UDF with the following functionality
 * Input -> member_id
 * Output -> Guest/Member

In [0]:
from pyspark.sql.functions import pandas_udf
import pandas as pd

@pandas_udf(returnType='string')
#input and output are pandas Series for this member function. but pandas_udf will return Pandas Series of a String
def member_type_pudf(id:pd.Series)-> pd.Series:
    return id.apply(lambda x: 'Guest' if x==0 else 'Member')

    

1.2 Use a Pandas UDF in Dataframe Transformations\
 List all bookings made by guests

In [0]:
bookings_df = spark.table("dev.spark_db.bookings")

bookings_df.where(member_type_pudf("member_id")=="Guest").display()

1.3. Register Pandas UDF for use in Spark SQL

In [0]:
spark.udf.register("get_member_type",member_type_pudf)

1.4 Pandas UDF from Spark SQL

In [0]:
%sql
select * from dev.spark_db.bookings
 where get_member_type(member_id)=="Guest"

1.5 Pandas UDF in Expressions

In [0]:
result_df = bookings_df.where("get_member_type(member_id)=='Guest'").display()

----------------------------------------------------------------------------------------------------------------------------------------------
###Python User Defined Table Function(UDTF)
 1. User-defined function that returns a table 
 2. Take Python Objects as input
 3. Operate one row at a time
 4. Serialized/Deserialized by pickle or Arrow 

####1. How to define and use a Python UDTF

1.1 Define a UDTF\
 Create a UDTF as the following.
 * Input: start_date (ex: 2025-07-27), expand_days (ex: 5)
 * Output: expand the given start_date to expand_days as below.
 ```
     +----------+
     |date_value|
     +----------+
     |2025-07-27|
     |2025-07-28|
     |2025-07-29|
     |2025-07-30|
     |2025-07-31|
     +----------+
 ```

In [0]:
from pyspark.sql.functions import udtf
from datetime import datetime, timedelta

@udtf(returnType="date_value: string",useArrow=True)
#udtf cannot be a plain function as udf, so we have to define a Class & udtf name should be the class name. this class can override a function called eval

class DateExploder:
    def eval(self, start_date: str, expand_days: int):
        current = datetime.strptime(start_date, "%Y-%m-%d")
        for i in range(expand_days):
            yield(current.strftime("%Y-%m-%d"), )
            current += timedelta(days=1)

In [0]:
from pyspark.sql.functions import lit

DateExploder(lit("2025-07-27"), lit(5)).display()  #udf or udtf expects a column so we can cannot pass value directly instead we can use lit to parse it as column

1.2 Use a UDTF in Dataframe Transformations

For using udtf , we need to a lateraljoin because udtf returns a table like output. and table can used in a join with a table and cannot be used withColumn or selectExpr

In [0]:
data_schema = "id int, name string, join_date string"
data_list = [(101, "Prashant", "2025-02-25"),
             (102, "Sushant", "2025-02-26")]

students_df = spark.createDataFrame(data_list, data_schema).alias("s")

result_df = (
    students_df.lateralJoin(DateExploder("s.join_date", lit(5)))
        .selectExpr("id", "name", "join_date", "date_value as attendance_date")
)

result_df.display()

1.3. Register UDTF for use in Spark SQL

In [0]:
spark.udtf.register("date_exploder", DateExploder)

1.4 UDTF from Spark SQL

In [0]:
#create a temporary view out of the dataframe
students_df.createOrReplaceTempView("students_view")

In [0]:
%sql
select * from students_view, lateral date_exploder(join_date, 2)